# 01a — Dataonderzoek en cleaning

Een geldige start kan bruikbaar zijn voor tellingen terwijl eindstation of duur onbruikbaar is. We onderzoeken schema, ontbrekende velden, duplicaten, foutieve waarden, extreme waarden en categorieën.

## Bijdragen — Messi10

**Raoul Buffels:** dataverwerking en cleaning, EDA en hypothesen, features, modellen,
tuning, evaluatie en documentatie; Citi Bike-frontend en backend.
**Jorre Van Dyck:** hulp bij deployment op Render.
Codex hielp met implementatie, uitleg, tests en uitvoering. Ieder teamlid moet de
eigen keuzes mondeling kunnen uitleggen.

We bepalen de projectmap en laden de kleine tabellen. Grote bronbestanden blijven op schijf.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations=[Path.cwd(),Path.cwd().parent,Path.cwd()/'Citybike_full']
ROOT=next(p for p in locations if (p/'citibike').exists())
sys.path.insert(0,str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
def report(name):
    return json.loads((REPORTS/name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS/name,dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


We laden het technische bronmanifest en ontwikkelingsplots. Op een nieuwe computer voer je eerst run.py prepare uit.

In [2]:
from citibike.datasets import manifest
from citibike.plots import show

## 2. Data discovery — OBTAIN

De code inventariseert bronnen en CSV-leden, inclusief maand-ZIPs in jaararchieven. SHA-256 identificeert de bronnen. Technische eindtestgegevens zoals schema en rij-aantallen mogen hier worden getoond; eindtestpatronen niet.

In [3]:
m = manifest()
display(pd.DataFrame([{'bronbestanden':m['source_count'],'bron_GiB':m['source_bytes']/2**30,'ruwe_rijen':m['raw_rows'],'bewaarde_rijen':m['retained_rows'],'laatste_run_minuten_inclusief_cachecontrole':m['processing_seconds']/60}]))
display(pd.DataFrame(m['coverage']).T)

,bronbestanden,bron_GiB,ruwe_rijen,bewaarde_rijen,laatste_run_minuten_inclusief_cachecontrole
0,175,29.604076,355199439,330714664,1.214597


,first_month,last_month,missing_months
NYC,2013-06,2026-08,[]
JC,2015-09,2026-08,[]


## 3. Schemaonderzoek

Historische headers verschillen in namen, hoofdletters en inhoud. We mappen start/stop, gebruiker en coördinaten naar één schema. Oudere bronnen bevatten geen ride-ID of fietstype. Station-ID’s blijven tekst; alleen de gedocumenteerde integer-`.0`-weergave in oude data wordt genormaliseerd.

In [4]:
display(pd.DataFrame([{'originele_kolommen':s['raw'],'canonieke_kolommen':s['canonical']} for s in m['schemas'].values()]))
display(Markdown(str(m['normalization'])))

,originele_kolommen,canonieke_kolommen
0,"[tripduration, starttime, stoptime, start stat...","[reported_duration_seconds, started_at, ended_..."
1,"[Trip Duration, Start Time, Stop Time, Start S...","[reported_duration_seconds, started_at, ended_..."
2,"[ride_id, rideable_type, started_at, ended_at,...","[ride_id, rideable_type, started_at, ended_at,..."


{'legacy_station_ids': 'remove integer .0 suffix only where legacy bike_id field exists; leading zeros preserved', 'coordinate_precision': '7 decimal places (~centimeter); equivalent source versions differ at floating-point serialization precision', 'overlap_decision': 'exact multiset equality after documented normalization; multiplicities preserved'}

## 4. Data assembly

Arrow leest batches van 8 MiB. DuckDB gebruikt maximaal 4 GiB en kan naar schijf spillen. Parquet bewaart één bruikbare kolomlaag. Bij overlappende bronversies vergelijkt de code de volledige canonieke rijmultisets in beide richtingen; gelijke historische ritten behouden hun multipliciteit.

In [5]:
parts = pd.DataFrame(m['partitions'])
display(parts.groupby('area')[['raw_rows','retained_rows','removed_rows','size_bytes','processing_seconds']].sum())
display(parts.loc[parts.overlap_removed > 0,['area','period','raw_rows','retained_rows','overlap_removed']])
display(Markdown(f"Hoogste gesamplede proces-RSS: {parts.peak_rss_bytes_sampled.max()/2**30:.2f} GiB. Dit is een gemeten ondergrens van de piek; het omvat niet elk kortdurend maximum."))

,raw_rows,retained_rows,removed_rows,size_bytes,processing_seconds
area,,,,,
JC,6911031,6897538,13493,190491560,77.086475
NYC,348288408,323817126,24471282,10861725181,4560.938800


,area,period,raw_rows,retained_rows,overlap_removed
132,NYC,2013-06,1155406,577702,577703
133,NYC,2013-07,1686832,843417,843416
134,NYC,2013-08,2003916,1001958,1001958
135,NYC,2013-09,2068718,1034359,1034359
136,NYC,2013-10,2075424,1037712,1037712
137,NYC,2013-11,1351548,675774,675774
138,NYC,2013-12,887932,443966,443966
187,NYC,2018-01,1437988,718994,718994
188,NYC,2018-02,1686228,843114,843114
189,NYC,2018-03,1953344,976672,976672


Hoogste gesamplede proces-RSS: 0.72 GiB. Dit is een gemeten ondergrens van de piek; het omvat niet elk kortdurend maximum.

We vergelijken gemeten geheugen en rekentijd. Gesamplede RSS mist mogelijk korte pieken en is geen totale workerpiek.

In [6]:
if (REPORTS/'resource_profile.json').exists():
    resources=pd.DataFrame(report('resource_profile.json')['processes'])
    resources['peak_RSS_GiB']=resources.peak_rss_bytes/2**30
    display(resources.groupby('stage')[['peak_RSS_GiB','samples']].max())
    display(Markdown(report('resource_profile.json')['note']))
display(Markdown(f"Som van gemeten afzonderlijke partitionbouwtijden: {parts.processing_seconds.sum()/60:.1f} minuten. De manifest-walltime is de laatste run inclusief cachecontrole; eerdere mislukte pogingen zijn geen onderdeel van die walltime."))

,peak_RSS_GiB,samples
stage,,
evaluate,0.890850,19
notebooks,0.048523,34
prepare,4.012024,3709
research,3.827644,1060
train,1.396580,3940
verify,2.087967,119


1-second sampled RSS of CLI main processes, not combined worker memory. Preparation monitoring started after early partitions; restarts can leave gaps. Not an exact continuous peak.

Som van gemeten afzonderlijke partitionbouwtijden: 77.3 minuten. De manifest-walltime is de laatste run inclusief cachecontrole; eerdere mislukte pogingen zijn geen onderdeel van die walltime.

De resourceprofielen meten RSS van CLI-hoofdprocessen, niet de gezamenlijke RSS van eventuele PyCaret/joblib-workers. Korte pieken kunnen tussen samples vallen. De DuckDB-limiet geldt voor intern DuckDB-geheugen en is geen limiet op alle Python-processen samen.

In [7]:
display(Markdown(f"Gemeten schrijfstappen voor de maandgrenscorrectie: {parts.get('assembly_seconds',pd.Series(dtype=float)).sum()/60:.1f} minuten. Globale controle, checksumtijd en eerdere onderbroken pogingen vallen buiten deze som; de logs en hervataudits leggen die afzonderlijk vast."))

Gemeten schrijfstappen voor de maandgrenscorrectie: 27.1 minuten. Globale controle, checksumtijd en eerdere onderbroken pogingen vallen buiten deze som; de logs en hervataudits leggen die afzonderlijk vast.

## 5. Cleaning — SCRUB

Een rit kan bruikbaar zijn voor vertrektellingen maar niet voor duur of geografie. We behouden flags, zodat ontbrekende eindstations geldige vertrekken niet automatisch verwijderen. Negatieve/nulduur is onbruikbaar voor duur. Ritten langer dan 24 uur worden gemarkeerd en onderzocht, niet automatisch verwijderd. Nullable onbekende historische velden zijn geen parsingfouten.

Maandarchieven bevatten soms vertrekrecords uit een andere maand. De technische assemblylaag deelt deze in op de werkelijke lokale vertrekdatum. Gelijke moderne ID-inhoud en volledig bewezen historische bronsegmentkopieën tellen eenmaal; conflicten stoppen verwerking. Identieke historische ritten binnen een bron behouden hun multipliciteit. De eindtestgrenzen volgen vertrekdatums, niet ZIP-namen. Originele maandcaches blijven bewaard voor betrouwbare cache-invalidation.

In [8]:
display(pd.DataFrame(m['month_boundary_audit']))
display(parts.loc[(parts.get('rows_moved_in',0)+parts.get('rows_moved_out',0)+parts.get('cross_partition_duplicates_removed',0))>0,['area','period','rows_moved_in','rows_moved_out','cross_partition_duplicates_removed']])

,area,modern_equivalent_copies_removed,station_id_serialization_alias_ids,timestamp_precision_alias_ids,timestamp_precision_proof,station_alias_proof,legacy_source_segment_proofs,legacy_equivalent_copies_removed,unproven_legacy_segments,actual_retained_timestamp_bounds,cross_id_conflicts,normalization,uncovered_boundary_rows
0,NYC,512,"[9C5F8DE7C994C151, E8186EECD7540A81, 2C9464568...",[],"JC only: one complete whole-second version, id...",fractional trailing-zero serialization only; s...,[],0,[{'source_path': 'C:/Deeplearning/CloudAI/City...,"[2013-06-01 00:00:01, 2026-08-31 23:58:01.765000]",0,actual local departure month; unknown coverage...,0
1,JC,42,[],"[09D67B6866E802DA, 15413FAB9CC9F156, 1AB4E466D...","JC only: one complete whole-second version, id...",fractional trailing-zero serialization only; s...,[{'source_path': 'C:/Deeplearning/CloudAI/City...,13451,[],"[2015-09-21 14:53:16, 2026-08-31 23:53:36.692000]",0,actual local departure month; unknown coverage...,0


,area,period,rows_moved_in,rows_moved_out,cross_partition_duplicates_removed
60,JC,2020-09,0,0,13451
104,JC,2024-05,24,0,24
105,JC,2024-06,13,24,0
106,JC,2024-07,10,13,0
107,JC,2024-08,15,10,0
...,...,...,...,...,...
285,NYC,2026-03,0,433,0
287,NYC,2026-05,635,0,512
288,NYC,2026-06,869,635,0
289,NYC,2026-07,1454,869,0


Kwaliteitsflags tonen welke velden bruikbaar zijn. Een missing veld verwijdert niet automatisch de hele rit.

In [9]:
quality = pd.DataFrame([{'area':p['area'],'period':p['period'],**p['quality_after']} for p in m['partitions']])
display(quality.groupby('area').sum(numeric_only=True).T)
display(parts[['area','period','raw_rows','retained_rows','removed_rows','removed_percent']].head(15))

area,JC,NYC
rows,6897538,323817126
valid_demand,6897538,323817126
invalid_start,0,0
invalid_end,0,0
outside_source_month,0,0
missing_start_station,205,89160
missing_end_station,21288,678311
bad_start_coordinates,5,85836
bad_end_coordinates,12471,607822
nonpositive_duration,238,1962


,area,period,raw_rows,retained_rows,removed_rows,removed_percent
0,JC,2015-09,6668,6668,0,0.0
1,JC,2015-10,19264,19264,0,0.0
2,JC,2015-11,15113,15113,0,0.0
3,JC,2015-12,11838,11838,0,0.0
4,JC,2016-01,7479,7479,0,0.0
5,JC,2016-02,8250,8250,0,0.0
6,JC,2016-03,13511,13511,0,0.0
7,JC,2016-04,16342,16342,0,0.0
8,JC,2016-05,19488,19488,0,0.0
9,JC,2016-06,23947,23947,0,0.0


Offsetloze tijden kunnen bij het herhaalde najaarsuur geen unieke fysieke ritduur geven. De statistische duurlaag gebruikt de oude expliciet gerapporteerde duur waar beschikbaar; anders het verschil na tijdzoneconversie. Ritten met een ambigu start- of einduur worden alleen uit fysieke-duurstatistiek uitgesloten, niet uit vertrektellingen. De oorspronkelijke klokduur en flags blijven in Parquet.

In [10]:
table('NYC_duration_time_quality.csv')
table('JC_duration_time_quality.csv')

,uncertain_physical_duration,usable_physical_duration
0,20400,303478007


,uncertain_physical_duration,usable_physical_duration
0,440,6471324


Concrete extreme ritten tonen waarom lange geregistreerde duur niet zomaar werkelijk fietsen betekent.

In [11]:
display(pd.DataFrame([report('NYC_duration_sample_audit.json'),report('JC_duration_sample_audit.json')]))
table('NYC_extreme_duration_examples.csv')

,area,development_only,sample_rows,sample_over_24h,sample_over_7days,median_minutes,maximum_minutes,interpretation,action,sampling_limitation
0,NYC,True,200000,104,16,9.883633,610844.077033,Extreme start/eindtijden kunnen weken of maand...,Positieve identificeerbare duur blijft behoude...,"Deterministische, begrensde steekproef; voorbe..."
1,JC,True,6674,4,0,6.550000,3942.216667,Extreme start/eindtijden kunnen weken of maand...,Positieve identificeerbare duur blijft behoude...,"Deterministische, begrensde steekproef; voorbe..."


,started_at,ended_at,duration_minutes,member_casual,rideable_type
0,2019-09-26 10:12:51.861,2020-11-23 14:56:56.483,610844.077033,casual,classic_bike
1,2020-07-13 15:34:13.720,2021-07-17 21:34:22.435,531720.145250,casual,classic_bike
2,2020-09-13 15:37:08.283,2021-04-14 11:32:53.012,306475.745483,casual,classic_bike
3,2020-09-06 19:09:01.372,2021-02-05 14:13:37.419,218584.600783,casual,classic_bike
4,2021-03-10 15:24:23.872,2021-05-16 13:25:17.222,96360.889167,casual,electric_bike
5,2019-09-05 11:41:08.418,2019-10-03 20:42:17.421,40861.150000,casual,unknown
6,2020-09-04 16:18:31.537,2020-09-28 23:36:59.382,34998.464083,member,electric_bike
7,2023-07-17 16:34:55.180,2023-08-08 13:21:15.435,31486.337583,casual,classic_bike
8,2020-07-22 17:41:44.747,2020-08-10 10:10:53.698,26909.149183,casual,classic_bike
9,2020-07-26 14:27:16.142,2020-08-11 22:14:17.515,23507.022883,casual,classic_bike


We tonen missing velden en de afzonderlijke duplicaatsteekproef; die steekproef bewijst geen volledige deduplicatie.

In [12]:
table('NYC_missing_values.csv')
display(pd.DataFrame([report('NYC_sample_duplicates.json')]))

,column,missing
0,ride_id,91944421
1,rideable_type,0
2,started_at,0
3,ended_at,0
4,start_station_id,77666
5,start_station_name,77666
6,end_station_id,622634
7,end_station_name,600940
8,start_lat,74342
9,start_lng,74342


,sample_rows,duplicate_sample_rows,interpretation
0,200000,0,Exact equality of selected sample fields is a ...


## Outliers en categorieën

Lange geregistreerde duur bewijst geen voortdurend fietsen. We markeren en onderzoeken deze waarden; een geldige start telt nog mee. Oude onbekende gebruikers- en fietscategorieën blijven zichtbaar als unknown.

In [13]:
show('duration')
table('NYC_duration_statistics.csv')
show('user_bike')
table('NYC_user_bike.csv')

<Figure size 1100x400 with 2 Axes>

De deterministische steekproef bevat 200,000 ritten. Mediaan: 9.88 minuten; IQR: 11.56 minuten; maximum: 610,844.1 minuten. Boxplot-uitbijters worden alleen verborgen in deze weergave, niet verwijderd uit de data. Exacte groepsaantallen en benaderde volledige-data-kwantielen staan in de aggregatietabellen.

,rides,mean,sd,quantiles,minimum,maximum
0,303478007,18.073241,818.247805,[ 5.76088778 9.90086991 17.28085877 36.364671...,0.016983,1.041872e+06


<Figure size 1000x400 with 1 Axes>

91,944,421 ritten hebben geen bekend fietstype, onder meer door het historische schema. Unknown wordt daarom niet als een echte fietscategorie geïnterpreteerd. Veranderingen in categorieën kunnen zowel gedrag als registratie weerspiegelen.

,member_casual,rideable_type,rides,mean_duration,median_duration
0,member,electric_bike,86921146,12.893676,8.929740
1,member,classic_bike,81848795,15.876694,8.690940
2,member,unknown,80797370,13.364979,9.524468
3,casual,electric_bike,22457891,24.392569,12.724321
4,casual,classic_bike,20326666,51.049436,16.411759
5,casual,unknown,11095271,35.901387,20.727488
6,unknown,unknown,51780,26.740441,20.040322


## Beslissingen voor de definitieve voorbereiding

Deze regels worden in notebook 02 via dezelfde modules toegepast. We verwijderen geen waarden omdat een modelscore daarmee beter lijkt.

In [14]:
display(pd.DataFrame([
 {'probleem':'bronkopie','beslissing':'alleen bewezen kopieën verwijderen; historische multipliciteit bewaren'},
 {'probleem':'missing eindstation','beslissing':'geldige start behouden voor tellingen'},
 {'probleem':'negatieve/nulduur','beslissing':'uitsluiten van duurstatistiek, flag behouden'},
 {'probleem':'duur >24 uur','beslissing':'markeren en onderzoeken, geen willekeurige cutoff'},
 {'probleem':'onbekende categorie','beslissing':'unknown behouden'},
 {'probleem':'DST ambiguïteit','beslissing':'alleen onbetrouwbare fysieke duur uitsluiten'},
 {'probleem':'ontbrekende maand','beslissing':'stoppen, niet als nul invullen'}
]))

,probleem,beslissing
0,bronkopie,alleen bewezen kopieën verwijderen; historisch...
1,missing eindstation,geldige start behouden voor tellingen
2,negatieve/nulduur,"uitsluiten van duurstatistiek, flag behouden"
3,duur >24 uur,"markeren en onderzoeken, geen willekeurige cutoff"
4,onbekende categorie,unknown behouden
5,DST ambiguïteit,alleen onbetrouwbare fysieke duur uitsluiten
6,ontbrekende maand,"stoppen, niet als nul invullen"
